In [0]:
%sql
-- Cria uma tabela Fato na camada Gold
CREATE OR REPLACE TABLE projeto_bcb.gold.fact_cotacoes
USING DELTA
AS
WITH min_max AS (
    SELECT MIN(Data) AS min_data, MAX(Data) AS max_data 
    FROM projeto_bcb.silver.cotacoes
),
datas_periodo AS (
    SELECT explode(sequence(min_data, max_data, interval 1 day)) AS Data 
    FROM min_max
),
moedas AS (
    SELECT DISTINCT Moeda 
    FROM projeto_bcb.bronze.dim_moedas
),
cross_tb AS (
    SELECT m.Moeda, d.Data 
    FROM moedas m 
    CROSS JOIN datas_periodo d
),
datas_faltantes AS (
    SELECT cr.Moeda, cr.Data
    FROM cross_tb cr
    LEFT JOIN projeto_bcb.silver.cotacoes c
    ON cr.Moeda = c.Moeda AND cr.Data = c.Data
    WHERE c.Data IS NULL
),
linhas_adicionadas AS (
    SELECT dt_falt.Moeda, dt_falt.Data, NULL AS Cotacao
    FROM datas_faltantes dt_falt
),
append AS (
    SELECT Moeda, Data, Cotacao FROM projeto_bcb.silver.cotacoes
    UNION ALL
    SELECT Moeda, Data, Cotacao FROM linhas_adicionadas
)
-- Preenche os nulos
SELECT 
    Moeda,
    Data,
    last_value(Cotacao, true) OVER (
        PARTITION BY Moeda 
        ORDER BY Data 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS Cotacao
FROM append
ORDER BY Data, Moeda;

In [0]:
%sql
SELECT * FROM projeto_bcb.gold.fact_cotacoes 
ORDER BY Moeda, Data DESC
LIMIT 10;